Group 4:
* Greatness Adeneye
* Karishma Nainwani
* Naila Farooqi

## Theme 3: Named Entity Recognition (NER) Using Transformers


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from transformers import BertTokenizerFast
import torch
import random


## Data Collection:

In [ ]:
# Load CoNLL files (returns list_of_sentences, list_of_label_seqs)
def load_conll_data(filepath):
    sentences = []     # list of token lists
    labels = []        # list of label lists
    tokens = []
    ner_tags = []
    with open(filepath, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                if tokens:
                    sentences.append(tokens)
                    labels.append(ner_tags)
                    tokens = []
                    ner_tags = []
                continue
            parts = line.split()
            # token is first column, NER tag is last column
            token = parts[0]
            tag = parts[-1]
            tokens.append(token)
            ner_tags.append(tag)
    # catch any trailing sentence not followed by newline
    if tokens:
        sentences.append(tokens)
        labels.append(ner_tags)
    return sentences, labels


In [ ]:
train_sentences, train_labels = load_conll_data("/content/train.txt")
valid_sentences, valid_labels = load_conll_data("/content/valid.txt")
test_sentences, test_labels = load_conll_data("/content/test.txt")

In [ ]:
print("Counts -> train:", len(train_sentences),
      "valid:", len(valid_sentences), "test:", len(test_sentences))

Counts -> train: 14987 valid: 3466 test: 3684


Label Mapping

In [ ]:
# Create label list and mappings
label_list = [
    "O",
    "B-PER", "I-PER",
    "B-ORG", "I-ORG",
    "B-LOC", "I-LOC",
    "B-MISC", "I-MISC"
]
label2id = {lab: i for i, lab in enumerate(label_list)}
id2label = {i: lab for lab, i in label2id.items()}


## Pre-processing:


Tokenization for Transformer NER

In [ ]:
# Tokenizer
MODEL_NAME = "bert-base-cased"
tokenizer = BertTokenizerFast.from_pretrained(MODEL_NAME)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

In [ ]:
MAX_LENGTH = 128

In [ ]:
# Align labels for a single tokenized sentence
def align_labels_for_sentence(word_ids, token_labels):
    """
    Aligns original token-level labels to subword tokens.
    Special tokens/padding -> -100 (ignored in loss).
    Converts B-X to I-X for subwords.
    """
    label_ids = []
    previous_word_idx = None
    for word_idx in word_ids:
        if word_idx is None:
            label_ids.append(-100)
        elif word_idx != previous_word_idx:
            # first subword → original label
            label_ids.append(label2id[token_labels[word_idx]])
        else:
            # subsequent subword → convert B-X to I-X
            orig_label = token_labels[word_idx]
            label_ids.append(label2id["I-" + orig_label[2:]] if orig_label.startswith("B-") else label2id[orig_label])
        previous_word_idx = word_idx

    return label_ids

In [ ]:
# -----------------------------
# Encode and align all sentences
def encode_and_align_labels(sentences, labels, tokenizer, max_length=MAX_LENGTH):
    """
    sentences: list of token lists
    labels: list of label lists
    Returns: dict with input_ids, attention_mask, labels
    """
    input_ids_list = []
    attention_mask_list = []
    labels_list = []

    for tokens, lab_seq in zip(sentences, labels):
        # Tokenize the sentence
        encoding = tokenizer(
            tokens,
            is_split_into_words=True,
            truncation=True,
            padding="max_length",
            max_length=max_length,
            return_attention_mask=True
        )

        # Align labels to subwords
        label_ids = align_labels_for_sentence(encoding.word_ids(), lab_seq)

        # Store encoded outputs
        input_ids_list.append(encoding["input_ids"])
        attention_mask_list.append(encoding["attention_mask"])
        labels_list.append(label_ids)

    return {
        "input_ids": input_ids_list,
        "attention_mask": attention_mask_list,
        "labels": labels_list
    }

In [ ]:
# Encode all splits
train_enc = encode_and_align_labels(train_sentences, train_labels, tokenizer)
valid_enc = encode_and_align_labels(valid_sentences, valid_labels, tokenizer)
test_enc  = encode_and_align_labels(test_sentences, test_labels, tokenizer)

In [ ]:
import zipfile

zip_path = "/content/Dataset.zip"  # name of your zip file

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall("Dataset")  # folder to extract into

print("Extraction complete!")


Extraction complete!


In [ ]:
from datasets import Dataset

In [ ]:
# Convert to Hugging Face Dataset objects (keeps lists; Trainer accepts Dataset)
train_dataset = Dataset.from_dict(train_enc)
valid_dataset = Dataset.from_dict(valid_enc)
test_dataset  = Dataset.from_dict(test_enc)


In [ ]:
print("Example input_ids length (should equal MAX_LENGTH):", len(train_dataset[0]["input_ids"]))
print("Example labels length (should equal MAX_LENGTH):", len(train_dataset[0]["labels"]))
print("Unique label ids in one example (excluding -100):", set([x for x in train_dataset[0]["labels"] if x != -100]))


Example input_ids length (should equal MAX_LENGTH): 128
Example labels length (should equal MAX_LENGTH): 128
Unique label ids in one example (excluding -100): {0}


It is already divided into training, testing and validating sets

## Model Creation and Training:

In [ ]:
!pip install transformers datasets seqeval

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 4.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.1/75.1 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 511.6/511.6 kB 24.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 150.3/150.3 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.9/193.9 kB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 72.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 242.4/242.4 kB 23.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 221.6/221.6 kB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 377.3/377.3 kB 31.7 MB/s eta 0:00:00
  Created wheel fo

In [ ]:
import torch
from transformers import BertForTokenClassification, Trainer, TrainingArguments, DataCollatorForTokenClassification
from datasets import Dataset
from seqeval.metrics import precision_score, recall_score, f1_score, classification_report

/usr/local/lib/python3.12/dist-packages/jax/_src/cloud_tpu_init.py:86: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(


Load pre-trained BERT for token classification

In [ ]:
NUM_LABELS = len(label_list)

model = BertForTokenClassification.from_pretrained(
    "bert-base-cased",
    num_labels=NUM_LABELS,
    id2label=id2label,
    label2id=label2id
)


model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

Some weights of BertForTokenClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
# Prepare data collator
data_collator = DataCollatorForTokenClassification(tokenizer)


Metrics for evaluation

In [ ]:
def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    # Convert IDs to label strings
    true_labels = [[id2label[l] for l in label_seq if l != -100] for label_seq in labels]
    true_preds  = [[id2label[pred] for (pred, l) in zip(pred_seq, label_seq) if l != -100]
                   for pred_seq, label_seq in zip(predictions, labels)]

    return {
        "precision": precision_score(true_labels, true_preds),
        "recall": recall_score(true_labels, true_preds),
        "f1": f1_score(true_labels, true_preds)
    }


Training Arguments

In [ ]:
training_args = TrainingArguments(
    output_dir="./ner_bert",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=100,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    optim="adamw_torch" # Specify a non-fused optimizer
)

Initialize Trainer

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)


/tmp/ipython-input-2514064032.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [ ]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss,Precision,Recall,F1
1,0.080700,0.067873,0.892642,0.935054,0.913356
2,0.027100,0.050666,0.928524,0.948043,0.938182
3,0.010700,0.051644,0.931560,0.951461,0.941405


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


TrainOutput(global_step=2811, training_loss=0.057879682006105666, metrics={'train_runtime': 633.6725, 'train_samples_per_second': 70.977, 'train_steps_per_second': 4.436, 'total_flos': 2937226745150208.0, 'train_loss': 0.057879682006105666, 'epoch': 3.0})

In [ ]:
metrics = trainer.evaluate(test_dataset)
print(metrics)


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


{'eval_loss': 0.1598641574382782, 'eval_precision': 0.8854595336076817, 'eval_recall': 0.9149539333805812, 'eval_f1': 0.8999651446497037, 'eval_runtime': 12.3848, 'eval_samples_per_second': 298.43, 'eval_steps_per_second': 18.652, 'epoch': 3.0}


## Evaluation:

1. Training Loss

Measures how well the model fits the training data.

It decreases steadily from 0.0794 → 0.0128, which indicates that the model is learning and fitting the training dataset very well.

2. Validation Loss

Measures how well the model performs on unseen data (validation set).

It decreased initially (0.0606 → 0.0533) but slightly increased in the last epoch (0.0579).

Small increase is normal and shows the model is not overfitting significantly.

3. Precision, Recall, F1

Precision (0.903 → 0.932): How many of the predicted entities were correct. High precision means few false positives.

Recall (0.939 → 0.952): How many of the actual entities were correctly predicted. High recall means few false negatives.

F1-score (0.921 → 0.942): Harmonic mean of precision and recall — overall measure of model accuracy for entity recognition.

Observation:

All metrics improve across epochs, showing the model is learning effectively.

F1-score of ~0.94 indicates strong performance on the NER task.

**eval_loss (0.1745)**

The token classification loss on the test set.

Slightly higher than validation loss (normal) because the model has never seen the test set.

**eval_precision (0.8775)**

~87.8% of predicted entities were correct (low false positives).

**eval_recall (0.9160)**

~91.6% of true entities were correctly predicted (low false negatives).

**eval_f1 (0.8963)**

Harmonic mean of precision and recall.

F1 ~0.90 → very good performance for a NER model on unseen data.

**eval_runtime, samples/sec, steps/sec**

Execution statistics for inference speed.

Shows model is reasonably fast for evaluation.